### We will need to generate a .py file out of this but I kept it as a notebook for easier testing
I included as much comments to clarify what I am doing - we can decide if we want to keep them or not in the final version


In [ ]:
# Imports for cleaning functions
import re
import numpy as np
import pandas as pd

# NLTK functions
import nltk
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk import bigrams, trigrams
from nltk.probability import FreqDist

# Downloading necessary NLTK resources if they are not already downloaded
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# Lowercasing funcction - simple but we will probably need it for our custom algorithm (similarity search)
def lowercase(text):
    return text.lower()

# a function for removing stop words
def remove_stop_words(text):
    # generate tokenized words from the text - NLTK will miss things up without tokens
    text = nltk.word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    celaned_text = []
    for word in text:
        if word not in stop_words:
            celaned_text.append(word)
    # converting the list to a string so we can replace the actual text with the cleaned text
    return ' '.join(celaned_text)


# function to extract n-grams from the text
def extract_ngrams(text, n_gram, lenght=10, remove_punctuation=False):
    # returns the most common n-grams in the text as a list 
    # n_gram can be 1 for unigrams, 2 for bigrams, and 3 for trigrams ...etc.
    # count is the number of n-grams to return, default is 10
    text = nltk.word_tokenize(lowercase(text))
    # keeps only words, letters and numbers
    if remove_punctuation:
        text = []
        for word in text:
            if word.isalnum():
                text.append(word)

    ngrams = nltk.ngrams(text, n_gram)
     # full dictionary of n-grams and their frequencies
    ngram_frequency = FreqDist(ngrams)
    # returning only the top number based on the lenght parameter above
    return ngram_frequency.most_common(lenght)

# ---------------------------------------------------- ----------------- ----------------- ----------------- ----------------- -----------------

# these functions are specific to the jobs dataset and may not be useful for the other two
# Note: I implemented these to take lists as inputs and we may need to update them or make sure we pass the user's input is passed as a list to these functions

# this works on the locations column
def extract_state_codes(locations):
    # Dictionary of state names and their abbreviations
    state_abbreviations = {
        "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
        "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
        "Florida": "FL", "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID",
        "Illinois": "IL", "Indiana": "IN", "Iowa": "IA", "Kansas": "KS",
        "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
        "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
        "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV",
        "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY",
        "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK",
        "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
        "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT",
        "Vermont": "VT", "Virginia": "VA", "Washington": "WA", "West Virginia": "WV",
        "Wisconsin": "WI", "Wyoming": "WY"
    }
    
    state_codes = []
    for s in locations:
        first = False
        second = False
        us = False
        for k in state_abbreviations.keys():
            if k in s:
                state_codes.append((state_abbreviations.get(k)))
                first = True
                break
        if not first:
            for v in state_abbreviations.values():
                if v in s:
                    state_codes.append(v)
                    second = True
                    break
        if not first and not second:
            state_codes.append('US')

    return state_codes

# this works on the salaries column to make sure all salareis are the same format
def get_salaries(sals):
    low_salaries = []
    high_salaries = []

    for range_str in sals:
        if len(str(range_str))>0:
            numbers = re.findall(r'[\$]?([\d,]+)', str(range_str))
          #  print(f"Processing: {range_str}, found numbers: {numbers}")

            if len(numbers) == 2:
                try:
                    low_salary = int(numbers[0].replace(',', ''))
                    high_salary = int(numbers[1].replace(',', ''))

                    if low_salary < 1000:
                        low_salary *= 8 * 5 * 52
                    low_salaries.append(low_salary)
                    
                    if high_salary < 1000:
                        high_salary *= 8 * 5 * 52
                    high_salaries.append(high_salary)
                    
                except ValueError as e:
                    print(f"ValueError: {e} for entry: {range_str}")
                    low_salaries.append(np.nan)
                    high_salaries.append(np.nan)

            elif len(numbers) == 1:
                try:
                    low_salary = int(numbers[0].replace(',', ''))
                    low_salaries.append(low_salary)
                    high_salaries.append(np.nan)
                except ValueError as e:
                    print(f"ValueError: {e}")
                    low_salaries.append(np.nan)
                    high_salaries.append(np.nan)
            else:
                low_salaries.append(np.nan)
                high_salaries.append(np.nan)               

        else:
            low_salaries.append(np.nan)
            high_salaries.append(np.nan)

    return low_salaries, high_salaries
    
# this reads salaries from description column
def get_salaries_description(descriptions):
    low_salaries = []
    high_salaries = []
    
    # regex pattern to match salaries
    salary_pattern = re.compile(r'\$?([\d,]+)(?:\s*-\s*\$?([\d,]+))')
    #cycles over all descriptions
    for description in descriptions:
        #if the job has no description
        if pd.isna(description) or not isinstance(description, str):
            low_salaries.append(np.nan)  
            high_salaries.append(np.nan)
            continue 
        # compare the job description with the regex
        if salary_pattern.search(description):
            # before the dash
            low_salary_str = salary_pattern.search(description).group(1)
            # after the dash
            high_salary_str = salary_pattern.search(description).group(2)

            try:
                # Convert low salary to int
                low_salary = int(low_salary_str.replace(',', '').strip())
                # Convert to annual if needed
                if low_salary < 1000:
                        low_salary *= 8 * 5 * 52
                low_salaries.append(low_salary)
                
                # Check if the high salary information is available, convert it to int
                if high_salary_str:
                    high_salary = int(high_salary_str.replace(',', '').strip())
                    if high_salary < 1000:
                        high_salary *= 8 * 5 * 52
                    high_salaries.append(high_salary)
                # append np.nan if no high salary information is available       -    Discuss with team if we need to impute salaries or just leave them as np.nan
                else:
                    high_salaries.append(np.nan)
            # if the conversion fails 
            except ValueError:
                low_salaries.append(np.nan)  
                high_salaries.append(np.nan)
        # if no description found or not a string, add np.nan
        else:
            low_salaries.append(np.nan)
            high_salaries.append(np.nan)

    return low_salaries, high_salaries # as lists to be added to the dataframe


# Merge salaries from both sources into one column

def merge_salaries(from_salaries , from_descriptions):
    result = []
    for i in range(len(from_salaries)):
        if not np.isnan(from_salaries[i]):
            result.append(from_salaries[i])
            continue
        result.append(from_descriptions[i])
    return result

# there are some functions in my test file that Inwill push later after I test them.



In [7]:
# testing the functions of some data
import pandas as pd
data = pd.read_csv('combined.csv')
data.head()

,job_title,job_location,job_salary,job_summary,is_AI,source
0,Data Science Manager,"Chicago, IL","$120,000 - $150,000 a year",Our client is hiring a Data Scientist to suppo...,1,chatgpt ai
1,AI Engineer,"Boston, MA","$120,000 - $150,000 a year",This role sits within a cross-functional analy...,1,chatgpt ai
2,Statistician - Behavioral Biology,"Silver Spring, MD 20910",NaN,Overview: About The Geneva Foundation ...,0,human
3,Data Science Lead,"Chicago, IL 60601",NaN,Pinnacle AI Labs is a world-class research and...,1,claude ai
4,Data Scientist,"Boston, MA 02124",NaN,The Opportunity MassMutual’s Advanced Analy...,0,human


In [25]:
# selecting one of the job summary texts to test the functions on
text = data['job_summary'][0]
print('actual text:--------------\n',text)

# testing the function to remove stop words
cleaned_text = remove_stop_words(text)

print('cleaned text:--------------\n', cleaned_text)
print('_'*150)

print('Testing n-grams:')
print(extract_ngrams(cleaned_text, 3, 5))

actual text:--------------
 Our client is hiring a Data Scientist to support ongoing initiatives.

Figure piece need. Evening themselves assume open. Southern yet PM coach seven PM partner. Simple easy else again plant force skill. Rise can race meet yourself. Theory question television many contain eat high. Science method whose practice it life state. Discussion oil tonight loss someone room section her. From inside animal future computer. Less year someone reveal now accept side figure. Guy glass natural accept. Him experience me community trip foreign president. Unit himself institution performance buy letter share.

What you'll do: build machine learning models analyze large datasets work with stakeholders deploy solutions into production

Law question practice. Tough from someone develop.
Result goal yet size much marriage. Series every good agency.
He trouble international product. Bad much mind minute study. Relate live long conference free factor generation thousand.
Southern 